# Network Anomaly Detection - Robust Pipeline

Questo notebook è adattato per funzionare con qualsiasi dataset di traffico di rete che hai già, anche se le etichette sono assenti o contengono una sola classe.

### Funzionalità:
- Caricamento automatico del file (modifica il nome se necessario)
- Riconoscimento automatico della colonna etichette (`label`, `attack`, `class`, ecc.)
- Preprocessing robusto: encoding, scaling, gestione valori mancanti
- Feature engineering di esempio
- Addestramento Isolation Forest per anomaly detection
- Valutazione automatica se le etichette sono presenti e valide
- Diagnostica chiara se le etichette non sono utilizzabili

**Modifica solo il nome del file nella cella di caricamento se necessario.**

# Anomaly Detection in Network Traffic

Questo notebook guida attraverso tutte le fasi per rilevare anomalie nel traffico di rete.

## 1. Import Required Libraries

In [7]:
# Import delle librerie necessarie
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, roc_auc_score

## 2. Load and Explore Network Traffic Data

In [8]:
# 1. Carica i dati di traffico di rete
# Sostituisci il nome del file se necessario
df = pd.read_csv('../data/network-training.csv')
print('Colonne:', df.columns.tolist())
print(df.head())

Colonne: ['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']
   id       dur proto service state  spkts  dpkts  sbytes  dbytes       rate  \
0   1  0.121478   tcp       -   FIN      6      4     258     172  74.087490   
1   2  0.649902   tcp       -   FIN     14     38     734   42014  78.473372   
2   3  1.623129   tcp       -   FIN      8     16     364   13186  14.170161   
3   4  1.681642   tcp     ftp   FIN     12     12     628     770  13.677108   
4   5  0.449454   tcp       -   FIN     10      6     534     268  33.3738

# 2. Identifica la colonna delle etichette (label/attack/class)
label_col = None
for col in df.columns:
    if 'label' in col.lower() or 'attack' in col.lower() or 'class' in col.lower():
        label_col = col
        break
if label_col:
    print(f"Colonna etichette trovata: {label_col}")
    print('Valori unici:', df[label_col].unique())
else:
    print('Nessuna colonna etichette trovata. Il notebook proseguirà solo con anomaly detection non supervisionata.')

In [9]:
# 3. Preprocessing robusto
if 'df' in globals() and 'label_col' in globals():
    for col in df.select_dtypes(include=['object']).columns:
        if col != label_col:
            df[col].fillna(df[col].mode()[0], inplace=True)
            df[col] = LabelEncoder().fit_transform(df[col])
    for col in df.select_dtypes(include=['float', 'int']).columns:
        df[col].fillna(df[col].median(), inplace=True)
    scaler = StandardScaler()
    num_cols = [c for c in df.select_dtypes(include=['float', 'int']).columns if c != label_col]
    df[num_cols] = scaler.fit_transform(df[num_cols])
else:
    print('Errore: assicurati di aver eseguito le celle di caricamento dati e identificazione etichette prima di questa.')

Errore: assicurati di aver eseguito le celle di caricamento dati e identificazione etichette prima di questa.


## 4. Feature Engineering

In [10]:
# 4. Feature engineering (esempio)
if 'df' in globals():
    if 'src_bytes' in df.columns and 'dst_bytes' in df.columns:
        df['bytes_ratio'] = df['src_bytes'] / (df['dst_bytes'] + 1)

## 5. Visualize Network Traffic Patterns

In [11]:
# 5. Addestra Isolation Forest
if 'df' in globals():
    X = df.drop(columns=[label_col]) if label_col else df.copy()
    model = IsolationForest(contamination=0.01, random_state=42)
    model.fit(X)
    df['anomaly'] = model.predict(X)

NameError: name 'label_col' is not defined

## 6. Train Anomaly Detection Model

In [ ]:
# 6. Valutazione automatica (se possibile)
if 'df' in globals() and label_col:
    # Conversione etichette se necessario
    if df[label_col].dtype == object:
        df[label_col] = df[label_col].map({'normal': 1, 'anomaly': 0, 'benign': 1, 'malicious': 0})
    df = df.dropna(subset=[label_col])
    df = df[df[label_col].isin([0, 1])]
    labels = df[label_col].astype(int)
    anomalies = df['anomaly'].loc[labels.index]
    if len(set(labels)) > 1:
        print(classification_report(labels, anomalies))
        print('ROC AUC:', roc_auc_score(labels, anomalies))
    else:
        print('Le etichette contengono una sola classe, impossibile valutare.')
else:
    print('Valutazione supervisionata non possibile: nessuna colonna etichette trovata.')

## 7. Evaluate Model Performance

In [ ]:
# 2. Identifica la colonna delle etichette (label/attack/class)
label_col = None
for col in df.columns:
    if 'label' in col.lower() or 'attack' in col.lower() or 'class' in col.lower():
        label_col = col
        break
if label_col:
    print(f"Colonna etichette trovata: {label_col}")
    print('Valori unici:', df[label_col].unique())
else:
    print('Nessuna colonna etichette trovata. Il notebook proseguirà solo con anomaly detection non supervisionata.')

Le etichette contengono una sola classe, impossibile valutare.


In [ ]:
# Pipeline robusta per anomaly detection
# 1. Carica i dati
net_df = pd.read_csv('../data/network-training.csv')
print('Colonne:', net_df.columns.tolist())
print(net_df.head())

# 2. Identifica la colonna delle etichette
label_col = None
for col in net_df.columns:
    if 'label' in col.lower() or 'attack' in col.lower() or 'class' in col.lower():
        label_col = col
        break
if label_col:
    print(f"Colonna etichette trovata: {label_col}")
    print('Valori unici:', net_df[label_col].unique())
else:
    print('Nessuna colonna etichette trovata. Il notebook proseguirà solo con anomaly detection non supervisionata.')

# 3. Preprocessing
for col in net_df.select_dtypes(include=['object']).columns:
    if col != label_col:
        net_df[col].fillna(net_df[col].mode()[0], inplace=True)
        net_df[col] = LabelEncoder().fit_transform(net_df[col])
for col in net_df.select_dtypes(include=['float', 'int']).columns:
    net_df[col].fillna(net_df[col].median(), inplace=True)
scaler = StandardScaler()
num_cols = [c for c in net_df.select_dtypes(include=['float', 'int']).columns if c != label_col]
net_df[num_cols] = scaler.fit_transform(net_df[num_cols])

# 4. Feature engineering (esempio)
if 'src_bytes' in net_df.columns and 'dst_bytes' in net_df.columns:
    net_df['bytes_ratio'] = net_df['src_bytes'] / (net_df['dst_bytes'] + 1)

# 5. Addestra Isolation Forest
X = net_df.drop(columns=[label_col]) if label_col else net_df.copy()
model = IsolationForest(contamination=0.01, random_state=42)
model.fit(X)
preds = model.predict(X)
net_df['anomaly'] = preds

# 6. Valutazione (se etichette presenti e almeno due classi)
if label_col:
    # Conversione etichette se necessario
    if net_df[label_col].dtype == object:
        net_df[label_col] = net_df[label_col].map({'normal': 1, 'anomaly': 0, 'benign': 1, 'malicious': 0})
    net_df = net_df.dropna(subset=[label_col])
    net_df = net_df[net_df[label_col].isin([0, 1])]
    labels = net_df[label_col].astype(int)
    anomalies = net_df['anomaly'].loc[labels.index]
    if len(set(labels)) > 1:
        print(classification_report(labels, anomalies))
        print('ROC AUC:', roc_auc_score(labels, anomalies))
    else:
        print('Le etichette contengono una sola classe, impossibile valutare.')
else:
    print('Valutazione supervisionata non possibile: nessuna colonna etichette trovata.')

Colonne: ['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']
   id       dur proto service state  spkts  dpkts  sbytes  dbytes       rate  \
0   1  0.121478   tcp       -   FIN      6      4     258     172  74.087490   
1   2  0.649902   tcp       -   FIN     14     38     734   42014  78.473372   
2   3  1.623129   tcp       -   FIN      8     16     364   13186  14.170161   
3   4  1.681642   tcp     ftp   FIN     12     12     628     770  13.677108   
4   5  0.449454   tcp       -   FIN     10      6     534     268  33.3738

/tmp/ipykernel_28231/1862359649.py:22: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  net_df[col].fillna(net_df[col].mode()[0], inplace=True)
/tmp/ipykernel_28231/1862359649.py:22: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=T

Le etichette contengono una sola classe, impossibile valutare.


---

**Note:**
- Se ricevi il messaggio "Le etichette contengono una sola classe, impossibile valutare.", significa che il tuo dataset non ha abbastanza varietà nelle etichette per una valutazione supervisionata. Puoi comunque usare la colonna `anomaly` per analisi non supervisionate.
- Se vuoi aiuto per bilanciare o selezionare i dati, forniscimi un esempio delle etichette presenti nel tuo file.